[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-10_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 10: Proactive Demand and Market Shaping

### Purpose and scope

This notebook explores how pricing, behavioral nudges, and partnerships can influence demand and affect revenue in a small service-capacity example. It uses deterministic scenario analysis: candidate strategies are evaluated using the supplied assumptions, and the best candidate is selected separately within each strategy category. The selected category-level choices are then combined and evaluated as one scenario.

The result is an illustrative strategy comparison, not a globally optimal solution to a joint optimization problem. The category-level selections are made independently, so the combined scenario is not guaranteed to maximize net revenue over every possible combination.

### Inputs and outputs

The inputs are an eight-period baseline demand pattern, internal capacity per period, assumed price elasticity, prices and discounts, nudge effects and costs, and partnership capacities and costs. The notebook reports scenario-level demand and revenue, the selected strategy in each category, and the net revenue of the combined selected scenario.

### Model assumptions

- Demand is deterministic and measured in customer-equivalents; fractional values are allowed because the elasticity formulas represent expected demand.
- Price response is linear in the percentage price change. Demand is not allowed to fall below zero.
- Internal capacity limits the number of customers served in each period. A capacity-sharing partnership adds its stated capacity to this limit.
- Nudge costs are incurred once for each selected nudge, and partnership integration costs are fixed costs. Capacity-sharing costs are charged per unit of partner capacity actually used.
- Nudge and partnership effects are simplified scenario assumptions, not empirically estimated causal effects. Reliability scores, segment-level differences, time horizon, labor costs, and several market-intelligence fields are stored but are not used in the calculations below.
- The example does not model customer welfare, competitor reactions, operational scheduling, uncertainty, or interactions beyond the explicit combined-scenario calculation.

### Analysis sequence

1. Define the demand, pricing, nudge, and partnership data structures.
2. Initialize the market scenario and its assumptions.
3. Evaluate the pricing, nudge, and partnership candidates by enumeration.
4. Select the best candidate within each category and evaluate their combined scenario.
5. Inspect the numerical summaries and visual comparisons.


## 1. Representing demand and market interventions

A demand pattern is a list indexed by time period: `base_demand[t]` is the baseline expected demand in period `t`. The data classes below collect the parameters used to describe demand response, prices, nudges, and partnerships. Grouping related values makes the assumptions explicit and keeps each intervention's inputs together.

The elasticity parameters are response coefficients. For example, a price elasticity of `-0.8` means that the linear approximation used here predicts a 0.8% demand decrease for a 1% price increase. This is a simplified local response model; it is not a general demand curve.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")
random.seed(42)
np.random.seed(42)


@dataclass
class DemandElasticity:
    """Demand-response parameters and a baseline demand pattern."""

    price_elasticity: float
    time_elasticity: float
    promotion_elasticity: float
    segment_elasticity: Dict[str, float]
    base_demand: List[float]


@dataclass
class PricingStrategy:
    """Parameters that define the available pricing rules."""

    base_price: float
    peak_premium: float
    off_peak_discount: float
    promotion_discount: float
    time_based_pricing: bool
    demand_based_pricing: bool
    competitive_pricing: bool


@dataclass
class BehavioralNudge:
    """Assumed effect, cost, duration, and target of a behavioral nudge."""

    nudge_type: str
    effectiveness: float
    cost_per_nudge: float
    duration_hours: int
    target_segments: List[str]


@dataclass
class StrategicPartnership:
    """Capacity or promotional resources supplied by a partner."""

    partner_name: str
    partnership_type: str
    capacity_available: int
    cost_per_unit: float
    reliability_score: float
    integration_cost: float

## 2. Initialize the market scenario

`StrategicOrchestrator` stores the inputs and creates the assumed market components. Its initialization methods are kept separate so that the demand-response, pricing, nudge, partnership, customer-segment, and competitor assumptions can each be inspected independently. Some stored fields are contextual data rather than active model inputs; the scope notes identify which quantities enter the calculations.

In [ ]:
class StrategicOrchestrator:
    """Store assumptions and evaluate deterministic market-shaping scenarios."""

    def __init__(
        self,
        base_demand: List[float],
        capacity: int,
        cost_params: Dict,
        time_horizon: float = 480.0,
    ):
        """Initialize the scenario inputs and assumed market components."""
        self.base_demand = list(base_demand)
        self.capacity = capacity
        self.cost_params = cost_params
        self.time_horizon = time_horizon

        self.demand_elasticity = self._initialize_demand_elasticity()
        self.pricing_strategy = self._initialize_pricing_strategy()
        self.behavioral_nudges = self._initialize_behavioral_nudges()
        self.strategic_partnerships = self._initialize_partnerships()

        self.market_trends = {}
        self.customer_segments = self._initialize_customer_segments()
        self.competitor_analysis = self._initialize_competitor_analysis()

        self.optimization_results = {}
        self.revenue_analysis = {}
        self.market_impact = {}

    def _initialize_demand_elasticity(self) -> DemandElasticity:
        """Return the assumed demand-response coefficients."""
        return DemandElasticity(
            price_elasticity=-0.8,
            time_elasticity=-0.3,
            promotion_elasticity=0.5,
            segment_elasticity={
                "price_sensitive": -1.2,
                "convenience_focused": -0.4,
                "quality_seeking": -0.2,
                "loyal_customers": -0.1,
            },
            base_demand=self.base_demand,
        )

    def _initialize_pricing_strategy(self) -> PricingStrategy:
        """Return the base price and the available price adjustments."""
        return PricingStrategy(
            base_price=50.0,
            peak_premium=0.3,
            off_peak_discount=0.2,
            promotion_discount=0.15,
            time_based_pricing=True,
            demand_based_pricing=True,
            competitive_pricing=True,
        )

    def _initialize_behavioral_nudges(self) -> List[BehavioralNudge]:
        """Return the candidate nudges and their assumed effects and costs."""
        return [
            BehavioralNudge(
                nudge_type="social_proof",
                effectiveness=0.25,
                cost_per_nudge=5.0,
                duration_hours=4,
                target_segments=["convenience_focused", "price_sensitive"],
            ),
            BehavioralNudge(
                nudge_type="scarcity",
                effectiveness=0.35,
                cost_per_nudge=8.0,
                duration_hours=2,
                target_segments=["quality_seeking", "loyal_customers"],
            ),
            BehavioralNudge(
                nudge_type="convenience",
                effectiveness=0.20,
                cost_per_nudge=3.0,
                duration_hours=6,
                target_segments=["convenience_focused"],
            ),
            BehavioralNudge(
                nudge_type="gamification",
                effectiveness=0.15,
                cost_per_nudge=10.0,
                duration_hours=8,
                target_segments=["price_sensitive", "convenience_focused"],
            ),
        ]

    def _initialize_partnerships(self) -> List[StrategicPartnership]:
        """Return the assumed capacity and promotional partnerships."""
        return [
            StrategicPartnership(
                partner_name="Local Cafe",
                partnership_type="capacity_sharing",
                capacity_available=3,
                cost_per_unit=25.0,
                reliability_score=0.9,
                integration_cost=1000.0,
            ),
            StrategicPartnership(
                partner_name="Co-working Space",
                partnership_type="cross_promotion",
                capacity_available=5,
                cost_per_unit=30.0,
                reliability_score=0.85,
                integration_cost=1500.0,
            ),
            StrategicPartnership(
                partner_name="Community Center",
                partnership_type="joint_marketing",
                capacity_available=2,
                cost_per_unit=20.0,
                reliability_score=0.8,
                integration_cost=500.0,
            ),
        ]

    def _initialize_customer_segments(self) -> Dict[str, Dict]:
        """Return illustrative segment shares and response attributes."""
        return {
            "price_sensitive": {
                "size": 0.35,
                "avg_frequency": 2.5,
                "price_sensitivity": -1.2,
                "promotion_responsiveness": 0.8,
            },
            "convenience_focused": {
                "size": 0.25,
                "avg_frequency": 3.2,
                "price_sensitivity": -0.4,
                "promotion_responsiveness": 0.4,
            },
            "quality_seeking": {
                "size": 0.25,
                "avg_frequency": 2.8,
                "price_sensitivity": -0.2,
                "promotion_responsiveness": 0.3,
            },
            "loyal_customers": {
                "size": 0.15,
                "avg_frequency": 4.5,
                "price_sensitivity": -0.1,
                "promotion_responsiveness": 0.2,
            },
        }

    def _initialize_competitor_analysis(self) -> Dict[str, Dict]:
        """Return illustrative competitor and market context."""
        return {
            "direct_competitors": {
                "count": 3,
                "avg_price": 45.0,
                "capacity_utilization": 0.75,
                "market_share": 0.4,
            },
            "indirect_competitors": {
                "count": 5,
                "avg_price": 35.0,
                "capacity_utilization": 0.6,
                "market_share": 0.3,
            },
            "market_growth_rate": 0.05,
            "price_competition_intensity": 0.7,
        }

In [ ]:
print("Initializing the strategic market scenario...")

# The eight entries represent consecutive one-hour periods from 09:00 to 17:00.
base_demand = [3, 4, 7, 6, 5, 4, 3, 2]
capacity = 8

cost_params = {
    "base_labor_cost": 25.0,
    "peak_labor_cost": 30.0,
    "off_peak_labor_cost": 20.0,
    "partnership_cost": 25.0,
    "marketing_cost": 100.0,
    "nudge_cost_per_unit": 5.0,
}

orchestrator = StrategicOrchestrator(
    base_demand=base_demand,
    capacity=capacity,
    cost_params=cost_params,
    time_horizon=480.0,
)

print("Scenario initialized:")
print(f"  Baseline demand by period: {base_demand}")
print(f"  Internal capacity per period: {capacity}")
print(
    "  Assumed price elasticity: "
    f"{orchestrator.demand_elasticity.price_elasticity}"
)
print(f"  Candidate nudges: {len(orchestrator.behavioral_nudges)}")
print(f"  Candidate partnerships: {len(orchestrator.strategic_partnerships)}")
print(f"  Customer segments described: {len(orchestrator.customer_segments)}")
print(f"  Peak baseline demand: {max(base_demand)}")

assert len(base_demand) == 8
assert all(demand >= 0 for demand in base_demand)
assert capacity > 0

## 3. Evaluate candidate strategies

### Scenario enumeration

The method evaluates a finite list of specified candidates rather than searching a continuous decision space. For each category, it computes the relevant outcome for every candidate and selects the candidate with the largest category-specific score: gross revenue for pricing, return on investment for nudges, and net benefit for partnerships. This is enumeration over the listed alternatives; it does not establish a global optimum over unlisted strategies or all cross-category combinations.

For a period with baseline demand $d_t$, price $p_t$, and base price $p_0$, the linear price-response assumption gives potential demand

$$
d_t^{(p)} = \max\left(0,\ d_t\left[1 + \epsilon_p\left(\frac{p_t}{p_0}-1\right)\right]\right),
\qquad
s_t^{(p)} = \min\left(d_t^{(p)}, C\right),
$$

where $\epsilon_p$ is `price_elasticity`, $C$ is internal capacity, and $s_t^{(p)}$ is the number of customers served. Period revenue is $p_t s_t^{(p)}$. Keeping potential demand separate from served demand makes the capacity limit explicit.

For a selected nudge set $N$, the illustrative combined nudge effect is

$$
e_N = \left(\sum_{n\in N} e_n\right)\left[1-0.1\left(|N|-1\right)\right],
\qquad
d_t^{(N)} = d_t(1+e_N),
$$

where $e_n$ is the assumed effectiveness of nudge $n$. The factor reduces the additive effect when multiple nudges are selected. Nudge return on investment is defined as $(R_N-K_N)/K_N$, where $R_N$ is incremental revenue at the base price and $K_N$ is nudge cost. A zero-cost selection, if introduced, would have an undefined ratio; the implementation reports zero when the cost is zero.

Partnership net benefit is incremental revenue minus partnership cost. Capacity-sharing partnerships can serve baseline demand above internal capacity; promotional partnerships increase potential demand but do not increase service capacity. The enumeration and arithmetic are implemented in the named methods below.

In [ ]:
def calculate_dynamic_pricing_impact(self, pricing_scenario: str) -> Dict:
    """Evaluate one price rule, accounting for price response and capacity."""
    adjusted_demand = []
    potential_demand = []
    prices = []
    revenues = []

    for period, demand in enumerate(self.base_demand):
        if pricing_scenario == "time_based":
            # Periods 2 and 3 are the two highest-demand periods in this example.
            if period in (2, 3):
                price_multiplier = 1.0 + self.pricing_strategy.peak_premium
            else:
                price_multiplier = 1.0 - self.pricing_strategy.off_peak_discount
        elif pricing_scenario == "demand_based":
            utilization = demand / self.capacity
            if utilization > 0.8:
                price_multiplier = 1.0 + self.pricing_strategy.peak_premium
            elif utilization < 0.5:
                price_multiplier = 1.0 - self.pricing_strategy.off_peak_discount
            else:
                price_multiplier = 1.0
        elif pricing_scenario == "competitive":
            competitor_price = self.competitor_analysis["direct_competitors"]["avg_price"]
            price_multiplier = competitor_price / self.pricing_strategy.base_price
        else:
            raise ValueError(f"Unknown pricing scenario: {pricing_scenario}")

        price = self.pricing_strategy.base_price * price_multiplier
        demand_change = self.demand_elasticity.price_elasticity * (
            price_multiplier - 1.0
        )
        potential = max(0.0, demand * (1.0 + demand_change))
        served = min(potential, self.capacity)

        potential_demand.append(potential)
        adjusted_demand.append(served)
        prices.append(price)
        revenues.append(served * price)

    return {
        "potential_demand": potential_demand,
        "adjusted_demand": adjusted_demand,
        "prices": prices,
        "revenues": revenues,
        "total_revenue": sum(revenues),
        "avg_price": float(np.mean(prices)),
        "total_customers": sum(adjusted_demand),
    }


def calculate_behavioral_nudge_impact(
    self,
    nudge_combination: List[str],
    demand_pattern: List[float] = None,
) -> Dict:
    """Evaluate selected nudges against a supplied or baseline demand pattern."""
    if demand_pattern is None:
        demand_pattern = self.base_demand
    if len(demand_pattern) != len(self.base_demand):
        raise ValueError("The demand pattern must match the modeled time periods.")

    selected_nudges = [
        nudge
        for nudge in self.behavioral_nudges
        if nudge.nudge_type in nudge_combination
    ]
    nudge_impact = sum(nudge.effectiveness for nudge in selected_nudges)
    if selected_nudges:
        nudge_impact *= 1.0 - 0.1 * (len(selected_nudges) - 1)
    nudge_cost = sum(nudge.cost_per_nudge for nudge in selected_nudges)

    potential_demand = [
        max(0.0, demand * (1.0 + nudge_impact))
        for demand in demand_pattern
    ]
    adjusted_demand = [
        min(demand, self.capacity)
        for demand in potential_demand
    ]
    baseline_served = [
        min(demand, self.capacity)
        for demand in demand_pattern
    ]
    additional_customers = sum(adjusted_demand) - sum(baseline_served)
    incremental_revenue = (
        additional_customers * self.pricing_strategy.base_price
    )
    roi = (
        (incremental_revenue - nudge_cost) / nudge_cost
        if nudge_cost > 0
        else 0.0
    )

    return {
        "potential_demand": potential_demand,
        "adjusted_demand": adjusted_demand,
        "nudge_impact": nudge_impact,
        "nudge_cost": nudge_cost,
        "additional_customers": additional_customers,
        "incremental_revenue": incremental_revenue,
        "roi": roi,
    }


def calculate_partnership_benefits(
    self,
    partnership_strategy: str,
    demand_pattern: List[float] = None,
) -> Dict:
    """Evaluate one partnership option against a supplied demand pattern."""
    if demand_pattern is None:
        demand_pattern = self.base_demand
    if len(demand_pattern) != len(self.base_demand):
        raise ValueError("The demand pattern must match the modeled time periods.")

    base_served = [min(demand, self.capacity) for demand in demand_pattern]
    base_revenue = sum(base_served) * self.pricing_strategy.base_price

    if partnership_strategy == "capacity_sharing":
        partners = [
            partner
            for partner in self.strategic_partnerships
            if partner.partnership_type == "capacity_sharing"
        ]
        additional_capacity = sum(
            partner.capacity_available for partner in partners
        )
        served_demand = sum(
            min(max(demand - self.capacity, 0.0), additional_capacity)
            for demand in demand_pattern
        )
        partnership_cost = sum(
            partner.cost_per_unit
            * min(partner.capacity_available, served_demand)
            for partner in partners
        )
        additional_revenue = served_demand * self.pricing_strategy.base_price
        unmet_demand = sum(
            max(demand - self.capacity, 0.0) for demand in demand_pattern
        )

        return {
            "additional_capacity": additional_capacity,
            "partnership_cost": partnership_cost,
            "additional_revenue": additional_revenue,
            "net_benefit": additional_revenue - partnership_cost,
            "unmet_demand_served": (
                served_demand / unmet_demand if unmet_demand > 0 else 1.0
            ),
        }

    if partnership_strategy in ("cross_promotion", "joint_marketing"):
        if partnership_strategy == "cross_promotion":
            partners = [
                partner
                for partner in self.strategic_partnerships
                if partner.partnership_type == "cross_promotion"
            ]
            demand_increase = 0.15 * len(partners)
            strategy_name = "promotion_boost"
        else:
            partners = [
                partner
                for partner in self.strategic_partnerships
                if partner.partnership_type == "joint_marketing"
            ]
            demand_increase = 0.1 * len(partners)
            strategy_name = "market_expansion"

        potential_demand = [
            max(0.0, demand * (1.0 + demand_increase))
            for demand in demand_pattern
        ]
        adjusted_demand = [
            min(demand, self.capacity) for demand in potential_demand
        ]
        additional_customers = sum(adjusted_demand) - sum(base_served)
        additional_revenue = (
            additional_customers * self.pricing_strategy.base_price
        )
        partnership_cost = sum(partner.integration_cost for partner in partners)

        return {
            strategy_name: demand_increase,
            "potential_demand": potential_demand,
            "adjusted_demand": adjusted_demand,
            "partnership_cost": partnership_cost,
            "additional_revenue": additional_revenue,
            "net_benefit": additional_revenue - partnership_cost,
            "demand_increase": demand_increase,
        }

    raise ValueError(f"Unknown partnership strategy: {partnership_strategy}")


def optimize_market_shaping_strategy(self) -> Dict:
    """Enumerate candidates, select category leaders, and evaluate their combination."""
    pricing_scenarios = ["time_based", "demand_based", "competitive"]
    pricing_results = {
        scenario: self.calculate_dynamic_pricing_impact(scenario)
        for scenario in pricing_scenarios
    }

    nudge_combinations = [
        ["social_proof"],
        ["scarcity"],
        ["convenience"],
        ["social_proof", "scarcity"],
        ["social_proof", "convenience"],
        ["scarcity", "convenience"],
        ["social_proof", "scarcity", "convenience"],
    ]
    nudge_results = {
        "+".join(combination): self.calculate_behavioral_nudge_impact(
            combination
        )
        for combination in nudge_combinations
    }

    partnership_strategies = [
        "capacity_sharing",
        "cross_promotion",
        "joint_marketing",
    ]
    partnership_results = {
        strategy: self.calculate_partnership_benefits(strategy)
        for strategy in partnership_strategies
    }

    best_pricing = max(
        pricing_results,
        key=lambda scenario: pricing_results[scenario]["total_revenue"],
    )
    best_nudge = max(
        nudge_results,
        key=lambda combination: nudge_results[combination]["roi"],
    )
    best_partnership = max(
        partnership_results,
        key=lambda strategy: partnership_results[strategy]["net_benefit"],
    )

    pricing_result = pricing_results[best_pricing]
    nudge_result = nudge_results[best_nudge]
    partnership_result = partnership_results[best_partnership]
    selected_nudges = [
        nudge
        for nudge in self.behavioral_nudges
        if nudge.nudge_type in best_nudge.split("+")
    ]
    combined_nudge_effect = sum(
        nudge.effectiveness for nudge in selected_nudges
    )
    if selected_nudges:
        combined_nudge_effect *= 1.0 - 0.1 * (len(selected_nudges) - 1)

    partnership_demand_increase = partnership_result.get("demand_increase", 0.0)
    if best_partnership == "capacity_sharing":
        partner_capacity = partnership_result["additional_capacity"]
    else:
        partner_capacity = 0

    combined_potential_demand = []
    combined_served_demand = []
    for period, base_demand in enumerate(self.base_demand):
        price_multiplier = (
            pricing_result["prices"][period]
            / self.pricing_strategy.base_price
        )
        price_response = 1.0 + self.demand_elasticity.price_elasticity * (
            price_multiplier - 1.0
        )
        potential = max(
            0.0,
            base_demand
            * price_response
            * (1.0 + combined_nudge_effect)
            * (1.0 + partnership_demand_increase),
        )
        served = min(potential, self.capacity + partner_capacity)
        combined_potential_demand.append(potential)
        combined_served_demand.append(served)

    combined_gross_revenue = sum(
        served * price
        for served, price in zip(combined_served_demand, pricing_result["prices"])
    )
    combined_strategy_cost = (
        nudge_result["nudge_cost"] + partnership_result["partnership_cost"]
    )
    combined_net_revenue = combined_gross_revenue - combined_strategy_cost

    return {
        "pricing_results": pricing_results,
        "nudge_results": nudge_results,
        "partnership_results": partnership_results,
        "best_pricing": best_pricing,
        "best_nudge": best_nudge,
        "best_partnership": best_partnership,
        "combined_potential_demand": combined_potential_demand,
        "combined_served_demand": combined_served_demand,
        "combined_gross_revenue": combined_gross_revenue,
        "combined_strategy_cost": combined_strategy_cost,
        "optimized_revenue": combined_net_revenue,
    }


StrategicOrchestrator.calculate_dynamic_pricing_impact = (
    calculate_dynamic_pricing_impact
)
StrategicOrchestrator.calculate_behavioral_nudge_impact = (
    calculate_behavioral_nudge_impact
)
StrategicOrchestrator.calculate_partnership_benefits = (
    calculate_partnership_benefits
)
StrategicOrchestrator.optimize_market_shaping_strategy = (
    optimize_market_shaping_strategy
)

## 4. Select and interpret the scenario results

The candidate lists are enumerated category by category. The combined scenario uses the selected pricing rule, nudge combination, and partnership strategy. It applies the price-response, nudge, and promotional demand factors to baseline demand, limits served demand by available capacity, computes revenue at the selected period prices, and subtracts the selected nudge and partnership costs.

The displayed `Net revenue` is therefore gross revenue less the modeled strategy costs. Baseline revenue is gross revenue at the base price with internal capacity limits and no intervention costs. Their difference is a scenario comparison under these assumptions. Because the three category leaders are selected independently, this combined scenario should not be interpreted as the globally best joint combination.

In [ ]:
print("Evaluating the market-shaping candidates...")
print("=" * 52)

start_time = time.time()
optimization_results = orchestrator.optimize_market_shaping_strategy()
elapsed_time = time.time() - start_time

baseline_served = [
    min(demand, orchestrator.capacity)
    for demand in orchestrator.base_demand
]
baseline_revenue = (
    sum(baseline_served) * orchestrator.pricing_strategy.base_price
)
net_revenue = optimization_results["optimized_revenue"]

print(f"Candidate evaluation completed in {elapsed_time:.6f} seconds.")
print("\nSelected category-level strategies:")
print(f"  Pricing rule: {optimization_results['best_pricing']}")
print(f"  Nudge combination: {optimization_results['best_nudge']}")
print(f"  Partnership strategy: {optimization_results['best_partnership']}")
print("\nCombined-scenario financial summary:")
print(
    "  Gross revenue: "
    f"${optimization_results['combined_gross_revenue']:.2f}"
)
print(
    "  Nudge and partnership costs: "
    f"${optimization_results['combined_strategy_cost']:.2f}"
)
print(f"  Net revenue: ${net_revenue:.2f}")
print(f"  Baseline gross revenue: ${baseline_revenue:.2f}")
print(f"  Difference from baseline: ${net_revenue - baseline_revenue:.2f}")

assert len(optimization_results["combined_served_demand"]) == len(base_demand)
assert all(
    served <= capacity
    + (
        optimization_results["partnership_results"][
            optimization_results["best_partnership"]
        ].get("additional_capacity", 0)
    )
    for served in optimization_results["combined_served_demand"]
)
assert np.isclose(
    optimization_results["optimized_revenue"],
    optimization_results["combined_gross_revenue"]
    - optimization_results["combined_strategy_cost"],
)

## 5. Visualize the candidate comparisons

The panels show different outcomes, so their vertical scales and meanings differ. Pricing is compared by gross revenue and average price; nudge combinations by return on investment; and partnerships by net benefit and incremental revenue. The demand panel compares baseline demand with the selected combined scenario's served demand. Its capacity line is the internal capacity; the selected capacity-sharing option, if any, adds partner capacity to the service limit.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Strategic Market-Shaping Scenario Analysis", fontsize=16)

# Pricing candidates: revenue and price use separate, labeled axes.
ax1 = axes[0, 0]
pricing_results = optimization_results["pricing_results"]
scenarios = list(pricing_results)
revenues = [pricing_results[name]["total_revenue"] for name in scenarios]
average_prices = [pricing_results[name]["avg_price"] for name in scenarios]
positions = np.arange(len(scenarios))
bar_width = 0.35

ax1.bar(
    positions - bar_width / 2,
    revenues,
    bar_width,
    label="Gross revenue ($)",
    color="#2ecc71",
    alpha=0.85,
)
ax1.set_ylabel("Gross revenue ($)")
ax1_twin = ax1.twinx()
ax1_twin.bar(
    positions + bar_width / 2,
    average_prices,
    bar_width,
    label="Average price ($)",
    color="#3498db",
    alpha=0.85,
)
ax1_twin.set_ylabel("Average price ($)")
ax1.set_xlabel("Pricing rule")
ax1.set_title("Pricing Scenario Comparison")
ax1.set_xticks(positions)
ax1.set_xticklabels(scenarios)
handles_left, labels_left = ax1.get_legend_handles_labels()
handles_right, labels_right = ax1_twin.get_legend_handles_labels()
ax1.legend(handles_left + handles_right, labels_left + labels_right, loc="best")

# Nudge return on investment is net incremental return divided by cost.
ax2 = axes[0, 1]
nudge_results = optimization_results["nudge_results"]
nudge_names = list(nudge_results)
roi_values = [nudge_results[name]["roi"] for name in nudge_names]
bar_colors = plt.cm.RdYlGn(
    [min(1.0, max(0.0, (value + 1.0) / 5.0)) for value in roi_values]
)
ax2.bar(nudge_names, roi_values, color=bar_colors, alpha=0.85)
ax2.axhline(0.0, color="black", linewidth=0.8)
ax2.set_title("Behavioral Nudge Return on Investment")
ax2.set_ylabel("Net return / nudge cost (ratio)")
ax2.set_xlabel("Nudge combination")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

# Partnership outcomes distinguish incremental revenue from net benefit.
ax3 = axes[1, 0]
partnership_results = optimization_results["partnership_results"]
partnership_names = list(partnership_results)
net_benefits = [
    partnership_results[name]["net_benefit"] for name in partnership_names
]
additional_revenues = [
    partnership_results[name]["additional_revenue"]
    for name in partnership_names
]
positions = np.arange(len(partnership_names))
ax3.bar(
    positions - bar_width / 2,
    net_benefits,
    bar_width,
    label="Net benefit ($)",
    color="#f39c12",
    alpha=0.85,
)
ax3.bar(
    positions + bar_width / 2,
    additional_revenues,
    bar_width,
    label="Incremental revenue ($)",
    color="#e74c3c",
    alpha=0.85,
)
ax3.set_xlabel("Partnership strategy")
ax3.set_ylabel("Amount ($)")
ax3.set_title("Partnership Scenario Outcomes")
ax3.set_xticks(positions)
ax3.set_xticklabels(partnership_names, rotation=15)
ax3.legend()
ax3.grid(axis="y", alpha=0.3)

# Served demand in the combined scenario respects the applicable capacity.
ax4 = axes[1, 1]
periods = np.arange(len(orchestrator.base_demand))
period_labels = [f"{9 + period}:00" for period in periods]
ax4.plot(
    periods,
    orchestrator.base_demand,
    "o-",
    label="Baseline demand",
    linewidth=2,
    markersize=6,
)
ax4.plot(
    periods,
    optimization_results["combined_served_demand"],
    "s-",
    label="Combined scenario: served demand",
    linewidth=2,
    markersize=6,
)
ax4.axhline(
    y=orchestrator.capacity,
    color="red",
    linestyle="--",
    label="Internal capacity",
    alpha=0.8,
)
selected_partnership = optimization_results["best_partnership"]
partner_capacity = optimization_results["partnership_results"][
    selected_partnership
].get("additional_capacity", 0)
if partner_capacity:
    ax4.axhline(
        y=orchestrator.capacity + partner_capacity,
        color="purple",
        linestyle=":",
        label="Capacity including partner",
        alpha=0.8,
    )
ax4.set_xlabel("Period start time")
ax4.set_ylabel("Customers per one-hour period")
ax4.set_title("Baseline Demand and Combined-Scenario Service")
ax4.set_xticks(periods)
ax4.set_xticklabels(period_labels)
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Selected candidate outcomes:")
best_pricing_result = pricing_results[optimization_results["best_pricing"]]
best_nudge_result = nudge_results[optimization_results["best_nudge"]]
best_partnership_result = partnership_results[
    optimization_results["best_partnership"]
]
print(
    f"  Pricing gross revenue: "
    f"${best_pricing_result['total_revenue']:.2f}"
)
print(f"  Nudge ROI: {best_nudge_result['roi']:.2f}")
print(f"  Nudge cost: ${best_nudge_result['nudge_cost']:.2f}")
print(
    "  Partnership net benefit: "
    f"${best_partnership_result['net_benefit']:.2f}"
)
print(
    "  Combined-scenario gross revenue: "
    f"${optimization_results['combined_gross_revenue']:.2f}"
)
print(
    "  Combined-scenario strategy costs: "
    f"${optimization_results['combined_strategy_cost']:.2f}"
)
print(f"  Combined-scenario net revenue: ${net_revenue:.2f}")
print(f"  Baseline gross revenue: ${baseline_revenue:.2f}")
print(f"  Difference from baseline: ${net_revenue - baseline_revenue:.2f}")

## 6. Interpretation and appropriate use

The outputs describe the consequences of the stated assumptions and candidate lists. A higher pricing-scenario revenue reflects the modeled combination of price response and capacity limits. A higher nudge return on investment means more modeled net incremental revenue per unit of nudge cost. Partnership net benefit accounts for its modeled incremental revenue and costs. These measures answer different questions and should not be compared as if they were the same objective.

The combined scenario is a useful accounting check on a selected set of category-level strategies: its served demand is capacity-limited, its gross revenue uses the selected period prices, and its net revenue subtracts the modeled nudge and partnership costs. It does not prove that this combination is best overall. A full joint search would require evaluating the cross-product of all pricing, nudge, and partnership candidates under one common objective; that is outside the scenario-selection procedure demonstrated here.

### Strengths and limitations

- **Strengths:** The scenario assumptions are explicit; candidate outcomes are directly inspectable; capacity limits and intervention costs are represented; and the selected combination is evaluated using a consistent revenue calculation.
- **Limitations:** Elasticities and intervention effects are assumed rather than estimated; outcomes are deterministic; and the analysis omits labor scheduling, customer-segment-specific demand, partner reliability, uncertainty, competitor response, and longer-term market effects.
- **Practical use:** This type of analysis can organize questions for planning in capacity-constrained services. Real decisions require validated demand-response estimates, operational constraints, and careful assessment of customer and stakeholder impacts.

### Further study

Natural extensions include estimating the response parameters from observed data, adding period-specific labor and service constraints, and testing how results change under plausible uncertainty. These additions would require new assumptions and data; they are not implemented in this example.